# 03. Automatic Mixed Precision (AMP), Autocast, and GradScaler

- **Category:** 13_precision_and_numerics
- **Lineage & Citations:** Micikevicius et al. (ICLR 2018, *Mixed Precision Training*); PyTorch Core Team (2020, *torch.cuda.amp*); NVIDIA Apex Documentation (2019).
- **Purpose:** Formulate the mechanics of Automatic Mixed Precision (AMP), autocast operation dispatch policies (GEMM whitelist vs softmax blacklist), dynamic gradient loss scaling (`GradScaler`), and correct gradient clipping ordering.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. Posit Quire Takum and Logarithmic Number Systems LNS](02_posit_quire_takum_and_logarithmic_number_systems_lns.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. FP8 E4M3 E5M2 FNUZ and Minifloats FP6 FP4](04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use `GradScaler` when training with BF16 (BF16 preserves FP32's 8-bit dynamic range, rendering loss scaling redundant); never clip gradients prior to unscaling (scaling artificially suppresses clipped gradient norms by $1/S$).

## Mathematical Foundations & Governing Equations

1. **Dynamic Loss Scaling & Unscaling**:
   $$\mathcal{L}_{\text{scaled}} = S \cdot \mathcal{L}(\theta) \implies \mathbf{G}_{\text{scaled}} = S \cdot \nabla_\theta \mathcal{L}, \quad \mathbf{G} = \frac{1}{S} \mathbf{G}_{\text{scaled}}$$
2. **Finite Gradient Checking & Scale Backoff**:
   $$\text{found\_inf} = \bigvee_i \left( \operatorname{isnan}(\mathbf{G}_i) \lor \operatorname{isinf}(\mathbf{G}_i) \right) \implies S_{\text{new}} = \begin{cases} S \times \text{backoff\_factor}, & \text{if found\_inf} \\ S \times \text{growth\_factor}, & \text{if } c \ge N_{\text{growth}} \\ S, & \text{otherwise} \end{cases}$$
3. **Gradient Unscaling and L2 Norm Clipping**:
   $$\text{coeff} = \min\left(1.0, \; \frac{\tau}{\|\mathbf{G}\|_2 + \epsilon}\right), \quad \mathbf{G}_{\text{clipped}} = \text{coeff} \cdot \mathbf{G}$$
4. **Autocast Partitioning**: Whitelist GEMMs execute in FP16/BF16; blacklist reductions (Softmax, LayerNorm) evaluate in FP32.

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/13_precision_and_numerics/03_automatic_mixed_precision_autocast_and_gradscaler.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def amp_autocast_matmul_softmax(x, w, b):
    z_half = x.half() @ w.half() + b.half()
    return torch.softmax(z_half.float(), dim=-1)

def dynamic_grad_scaler_step(grads, scale, growth_tracker, growth_factor=2.0, backoff_factor=0.5, growth_interval=4):
    has_inf_nan = bool(torch.isinf(grads).any() or torch.isnan(grads).any())
    unscaled = grads * (1.0 / float(scale))
    if has_inf_nan:
        new_scale = float(scale) * float(backoff_factor)
        new_tracker = 0
        step_skipped = True
    else:
        new_tracker = int(growth_tracker) + 1
        if new_tracker >= int(growth_interval):
            new_scale = float(scale) * float(growth_factor)
            new_tracker = 0
        else:
            new_scale = float(scale)
        step_skipped = False
    return unscaled, new_scale, new_tracker, step_skipped

def unscale_and_clip_grads(scaled_grads, scale, max_norm=1.0, eps=1e-6):
    unscaled = scaled_grads * (1.0 / float(scale))
    total_norm = float(torch.norm(unscaled))
    coeff = min(1.0, float(max_norm) / (total_norm + float(eps)))
    return unscaled * coeff, total_norm

In [ ]:
# 1. Linear Anchor & AMP Matmul-Softmax Parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6
probs = amp_autocast_matmul_softmax(weights["amp_x"], weights["amp_w"], weights["amp_b"])
assert torch.max(torch.abs(probs - weights["amp_probs_ref"])) < 1e-5

# 2. Dynamic GradScaler Clean Step (Growth)
u_clean, s_clean, t_clean, skip_clean = dynamic_grad_scaler_step(
    weights["clean_grads"], scale=65536.0, growth_tracker=3, growth_factor=2.0, backoff_factor=0.5, growth_interval=4
)
assert torch.max(torch.abs(u_clean - weights["unscaled_clean_ref"])) < 1e-5
assert abs(s_clean - float(weights["scale_clean_ref"])) < 1e-5 and t_clean == int(weights["tracker_clean_ref"]) and not skip_clean

# 3. Dynamic GradScaler Overflow Step (Backoff)
over_g = weights["clean_grads"].clone()
over_g[1, 2] = float("inf")
u_over, s_over, t_over, skip_over = dynamic_grad_scaler_step(
    over_g, scale=65536.0, growth_tracker=2, growth_factor=2.0, backoff_factor=0.5, growth_interval=4
)
assert abs(s_over - float(weights["scale_over_ref"])) < 1e-5 and t_over == int(weights["tracker_over_ref"]) and skip_over

# 4. Unscale and Clip Gradients Parity
clipped, norm_val = unscale_and_clip_grads(weights["clip_scaled_grads"], scale=1000.0, max_norm=1.0)
assert torch.max(torch.abs(clipped - weights["clipped_ref"])) < 1e-5
assert abs(norm_val - float(weights["norm_ref"])) < 1e-5
print("All AMP, GradScaler, and Gradient Clipping assertions verified successfully!")

In [ ]:
# Microbenchmark: Autocast Forward + Scaler Step Latency
grads = weights["clean_grads"]
t0 = time.perf_counter()
for _ in range(500):
    _ = dynamic_grad_scaler_step(grads, scale=65536.0, growth_tracker=1)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"GradScaler step latency: {lat_us:.2f} us")

t0 = time.perf_counter()
for _ in range(500):
    _ = unscale_and_clip_grads(weights["clip_scaled_grads"], scale=1000.0, max_norm=1.0)
clip_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Unscale and Clip latency: {clip_us:.2f} us")

In [ ]:
# Pathological Misuse: Clipping Scaled Gradients before Unscaling
scaled_g = weights["clip_scaled_grads"]
scale_val = 1000.0
max_norm = 1.0

# Correct Order: Unscale -> Clip
correct_clipped, true_norm = unscale_and_clip_grads(scaled_g, scale=scale_val, max_norm=max_norm)

# Erroneous Order: Clip -> Unscale
scaled_norm = float(torch.norm(scaled_g))
erroneous_coeff = min(1.0, max_norm / (scaled_norm + 1e-6))
erroneous_clipped = (scaled_g * erroneous_coeff) * (1.0 / scale_val)

# Demonstrate that erroneous ordering suppresses gradients by factor of S (1000x)
ratio = float(torch.norm(correct_clipped) / torch.norm(erroneous_clipped))
print(f"True gradient norm: {true_norm:.2f}, Scaled gradient norm: {scaled_norm:.2f}")
print(f"Gradient suppression ratio (Correct / Erroneous): {ratio:.1f}x (Matches scale factor S = {scale_val:.0f})")
assert ratio > 900.0, "Erroneous clipping order must suppress gradient norm by factor of scale!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Autocast Policies**: GEMM and Conv operations run with high hardware throughput in FP16/BF16, while reductions and normalizations remain strictly in FP32 to prevent numerical underflow and catastrophic cancellation.
2. **Dynamic Loss Scaling**: Shifts small backpropagated gradients into the representable range of FP16 ($> 2^{-14}$), scaling back down and skipping weight updates upon detecting non-finite gradients (`inf`/`NaN`).
3. **Clipping Invariant**: Always unscale gradients before calculating gradient norms or executing gradient clipping, preventing devastating gradient suppression by $1/S$.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_automatic_mixed_precision_autocast_and_gradscaler.md](../../markdown/13_precision_and_numerics/03_automatic_mixed_precision_autocast_and_gradscaler.md)
- **PyTorch 2.x Module:** [03_automatic_mixed_precision_autocast_and_gradscaler.py](../../pytorch2/13_precision_and_numerics/03_automatic_mixed_precision_autocast_and_gradscaler.py)
- **Functional JAX Module:** [03_automatic_mixed_precision_autocast_and_gradscaler.py](../../jax/13_precision_and_numerics/03_automatic_mixed_precision_autocast_and_gradscaler.py)
- **Native Rust Candle Track:** [03_automatic_mixed_precision_autocast_and_gradscaler.rs](../../candle/13_precision_and_numerics/03_automatic_mixed_precision_autocast_and_gradscaler.rs)
- **Hardware Kernel (CPU / WGSL):** [03_automatic_mixed_precision_autocast_and_gradscaler.rs](../../kernels/13_precision_and_numerics/03_automatic_mixed_precision_autocast_and_gradscaler.rs)
- **Golden SafeTensors:** [03_automatic_mixed_precision_autocast_and_gradscaler.safetensors](../../weights/13_precision_and_numerics/03_automatic_mixed_precision_autocast_and_gradscaler.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. Posit Quire Takum and Logarithmic Number Systems LNS](02_posit_quire_takum_and_logarithmic_number_systems_lns.ipynb) | [Master Index](../../README.md#13_precision_and_numerics) | [Next: 04. FP8 E4M3 E5M2 FNUZ and Minifloats FP6 FP4](04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.ipynb) &rarr;